# Extended FSMs (guards and context)

An *extended* FSM adds data variables (context) and **guards**: boolean conditions that decide whether a transition may fire. Actions then update the data. This allows a small set of states to represent unbounded behaviour (here, any credit amount).

`VendingMachine` has states `idle` and `has_credit`, a `credit` counter, and a `price` of 75.

```mermaid
stateDiagram-v2
    [*] --> idle
    idle --> has_credit: insert [amount > 0]
    has_credit --> has_credit: insert [amount > 0]
    has_credit --> idle: select [credit == price]
    has_credit --> has_credit: select [credit >= price] / dispense
    has_credit --> has_credit: select [credit < price] / reject
    has_credit --> idle: refund / credit = 0
```

In [1]:
from petrilab.fsm.extended import VendingMachine


def snap(sm):
    return {
        "state": next(iter(sm.configuration)).id,
        "credit": sm.credit,
        "dispensed": sm.dispensed,
        "rejected": sm.rejected,
    }


sm = VendingMachine()
snap(sm)

{'state': 'idle', 'credit': 0, 'dispensed': 0, 'rejected': 0}

A new machine starts in `idle` with no credit.

## Underfunded selection

In [2]:
sm = VendingMachine()
sm.send("insert", amount=50)
sm.send("select")
snap(sm)

{'state': 'has_credit', 'credit': 50, 'dispensed': 0, 'rejected': 1}

50 < 75, so the `unless="enough"` branch fires: the machine stays in `has_credit`, the credit is kept, and `rejected` is incremented.

## Exact payment and change

In [3]:
exact = VendingMachine()
exact.send("insert", amount=75)
exact.send("select")

change = VendingMachine()
change.send("insert", amount=100)
change.send("select")
snap(exact), snap(change)

({'state': 'idle', 'credit': 0, 'dispensed': 1, 'rejected': 0},
 {'state': 'has_credit', 'credit': 25, 'dispensed': 1, 'rejected': 0})

Exact payment returns to `idle`. Overpaying dispenses and retains 25 of credit in `has_credit`.

## Accumulation and refund

In [4]:
sm = VendingMachine()
sm.send("insert", amount=25)
sm.send("insert", amount=25)
before = snap(sm)
sm.send("refund")
before, snap(sm)

({'state': 'has_credit', 'credit': 50, 'dispensed': 0, 'rejected': 0},
 {'state': 'idle', 'credit': 0, 'dispensed': 0, 'rejected': 0})

Insertions accumulate (to 50); `refund` resets the credit to 0 and returns to `idle`.

## Guards that reject events
Event parameters (`amount`) reach guards and actions. A failing guard means no transition applies, so the event is rejected.

In [5]:
from statemachine.exceptions import TransitionNotAllowed

sm = VendingMachine()
for event, kwargs in [("insert", {"amount": 0}), ("select", {})]:
    try:
        sm.send(event, **kwargs)
    except TransitionNotAllowed:
        print(f"{event} {kwargs} rejected in state", next(iter(sm.configuration)).id)
sm.credit

insert {'amount': 0} rejected in state idle
select {} rejected in state idle


0

A non-positive insert fails its guard and `select` has no transition from `idle`; the credit is untouched.